### 1 Charger les datasets enrichis

In [ ]:
import pandas as pd
from datasets import Dataset

DATA_PATH = "data/processed"

train_df = pd.read_csv(f"{DATA_PATH}/train_data.csv")
val_df = pd.read_csv(f"{DATA_PATH}/val_data.csv")
test_df = pd.read_csv(f"{DATA_PATH}/test_data.csv")

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

### 2- Charger ClinicalT5

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

MODEL_NAME = "medicalai/ClinicalT5-base"

tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    device_map="auto"
)

### 3- Ajouter LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

### 4- Tokenization

In [ ]:
MAX_INPUT = 512
MAX_TARGET = 128

def preprocess(example):

    model_inputs = tokenizer(
        example["prompt_enriched"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

### 5- Tokeniser les datasets

In [ ]:
train_dataset = train_dataset.map(preprocess, batched=False)
val_dataset = val_dataset.map(preprocess, batched=False)

### 6- Configurer l'entraînement

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="clinical_t5_lora",

    learning_rate=2e-4,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=3,

    logging_steps=100,

    evaluation_strategy="steps",
    eval_steps=1000,

    save_steps=1000,

    save_total_limit=2,

    fp16=True,

    report_to="none"
)

### 7- Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer
)

trainer.train()

### 8- Sauvegarder le modèle

In [ ]:
trainer.save_model("clinical_t5_lora_model")
tokenizer.save_pretrained("clinical_t5_lora_model")

### Générer un résumé

In [ ]:
def generate_summary(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = model.generate(
        **inputs,
        max_length=150
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)